---
title: "DRG Cleaning (Python) v2"

author: "Carlos Resurreccion"

date: "2024-11-19"

---


In [7]:
import os
import hashlib
import subprocess
import base64

TO_DEBUG = False  # Set to True to enable debug logs


def debug_print(message):
    """Print debug messages if TO_DEBUG is True."""
    if TO_DEBUG:
        print(message)


def hash_file(file_path):
    """Calculate the base64-encoded MD5 hash of a local file."""
    with open(file_path, "rb") as f:
        md5 = hashlib.md5()
        while chunk := f.read(8192):
            md5.update(chunk)
    # Convert the hex digest to base64
    return base64.b64encode(md5.digest()).decode('utf-8')


def hash_folder(folder_path):
    """Hash all files in a folder sequentially."""
    file_paths = []

    # Collect all file paths
    for root, _, files in os.walk(folder_path):
        for file in sorted(files):  # Sort files for consistency
            file_path = os.path.join(root, file)
            rel_path = os.path.relpath(file_path, folder_path)
            file_paths.append((file_path, rel_path))

    debug_print(f"Processing {len(file_paths)} files sequentially.")

    # Generate hashes
    file_hashes = {}
    for file_path, rel_path in file_paths:
        try:
            file_hashes[rel_path] = hash_file(file_path)
        except Exception as e:
            debug_print(f"Error hashing file {rel_path}: {e}")

    return file_hashes


def fetch_gcs_hashes(gcs_folder):
    """Fetch GCS file hashes using gsutil and return as a dictionary."""
    gcs_hashes = {}
    debug_print(f"Fetching GCS hashes for {gcs_folder}...")

    # Run the gsutil command
    result = subprocess.run(
        ["gsutil", "ls", "-L", "-r", gcs_folder],
        capture_output=True,
        text=True
    )

    # Check for errors
    if result.returncode != 0:
        debug_print(f"Error fetching GCS hashes: {result.stderr}")
        return gcs_hashes

    # Process the gsutil output
    lines = result.stdout.splitlines()
    current_file = None

    for line in lines:
        # Normalize the line to strip tabs and leading/trailing spaces
        line = line.strip()

        if line.startswith("gs://"):
            # If the line starts with "gs://", it indicates a new file
            current_file = line.rstrip(":")  # Remove the trailing colon
        elif "Hash (md5):" in line and current_file:
            # Extract the MD5 hash and associate it with the file
            md5_hash = line.split(":", 1)[1].strip()
            # Compute the relative path of the file
            relative_path = current_file.replace(gcs_folder + "/", "")
            gcs_hashes[relative_path] = md5_hash
            current_file = None  # Reset for the next file

    debug_print("Parsed GCS hashes:")
    for file, hash_value in gcs_hashes.items():
        debug_print(f"File: {file}, GCS Hash: {hash_value}")

    return gcs_hashes


def compare_hashes(local_hashes, gcs_hashes):
    """Compare local and GCS hashes and return mismatched files."""
    mismatched_files = []
    for file_path, local_hash in local_hashes.items():
        gcs_hash = gcs_hashes.get(file_path)
        debug_print(f"File: {file_path}")
        debug_print(f"Local Hash: {local_hash}")
        debug_print(f"GCS Hash: {gcs_hash}")
        if gcs_hash != local_hash:
            mismatched_files.append(file_path)
    return mismatched_files


def sanitize_gcs_path(path):
    """Normalize GCS paths."""
    if not path.startswith("gs://"):
        raise ValueError(f"Invalid GCS path: {path}")
    prefix = "gs://"
    return prefix + path[len(prefix):].replace("//", "/").rstrip("/")


def upload_files(local_folder, gcs_folder, mismatches, all_files):
    """Upload mismatched files or the entire directory to GCS, preserving directory structure."""
    gcs_folder = sanitize_gcs_path(gcs_folder)

    if set(mismatches) == set(all_files):
        debug_print(f"Uploading entire folder: {local_folder} to {gcs_folder}")
        subprocess.run(["gsutil", "-m", "cp", "-r", f"{local_folder}/*", gcs_folder], check=True)
    else:
        debug_print(f"Uploading {len(mismatches)} mismatched files to {gcs_folder}...")

        for file in mismatches:
            local_file_path = os.path.join(local_folder, file)
            destination_path = os.path.join(gcs_folder, file).replace("\\", "/")  # Ensure correct GCS path format
            try:
                subprocess.run(["gsutil", "cp", local_file_path, destination_path], check=True)
                debug_print(f"Uploaded: {local_file_path} -> {destination_path}")
            except subprocess.CalledProcessError as e:
                debug_print(f"Failed to upload {local_file_path}: {e.stderr}")


def download_files(local_folder, gcs_folder, mismatches):
    """Download only mismatched files from GCS."""
    gcs_folder = sanitize_gcs_path(gcs_folder)
    for file in mismatches:
        gcs_file_path = os.path.join(gcs_folder, file)
        local_file_path = os.path.join(local_folder, file)
        os.makedirs(os.path.dirname(local_file_path), exist_ok=True)
        debug_print(f"Downloading: {file}")
        subprocess.run(["gsutil", "cp", gcs_file_path, local_file_path], check=True)


def main():
    bucket_path = "gs://phic-claims-checkpoints/temp/data"
    vm_path = "/mnt/data-disk/data"
    desktop_path = "/home/data"

    # Flags for actions
    to_upload_from_vm = False
    to_upload_from_desktop = False
    to_download_to_desktop = False
    to_download_to_vm = False

    # Check if all flags are False; if so, exit the script
    if not any([to_upload_from_vm, to_upload_from_desktop, to_download_to_desktop, to_download_to_vm]):
        debug_print("All flags are set to False. Skipping script execution.")
        return

    # Determine the base path
    base_path = desktop_path if to_upload_from_desktop or to_download_to_desktop else vm_path

    for folder in ["md5", "aux-files", "checkpoints", "partial-claims", "sampled-claims", "profvis"]:
        local_folder = os.path.join(base_path, folder)
        gcs_folder = os.path.join(bucket_path, folder)

        if not os.path.exists(local_folder) or not any(os.scandir(local_folder)):
            debug_print(f"Skipping empty folder: {folder}")
            continue

        debug_print(f"Checking folder: {folder}")

        # Hash local and GCS files
        local_hashes = hash_folder(local_folder)
        gcs_hashes = fetch_gcs_hashes(gcs_folder)

        # Find mismatched files
        mismatches = compare_hashes(local_hashes, gcs_hashes)
        all_files = list(local_hashes.keys())
        
        if mismatches:
            debug_print(f"Found {len(mismatches)} mismatched files in {folder}.")
            if to_upload_from_vm or to_upload_from_desktop:
                upload_files(local_folder, gcs_folder, mismatches, all_files)
            if to_download_to_vm or to_download_to_desktop:
                download_files(local_folder, gcs_folder, mismatches)
        else:
            debug_print(f"No mismatches found for folder: {folder}.")

    debug_print("Sync completed.")


if __name__ == "__main__":
    main()

In [8]:
# Initialize variables
thread_offset = 0
sample_size_divisor = 625

# Whether to sample each split_part by sample_size_divisor
# (useful when iterating through code runs in quick succession)
to_sample = False  # Flag to indicate sampling
to_write = True    # Flag to enable writing outputs
to_flush = False   # Flag to enable flushing buffers
to_parallel = True # Flag for enabling parallel processing
to_debug = False   # Flag for enabling debugging

# Display the parallelization status
print("Parallelization:", to_parallel, "\n")

# Set verbose output based on debugging flag
verbose_output = True if to_debug else False

# Path to the year_to_load file
year_file_path = "/home/resurreccion_cmc_gmail_com/drg-pipeline/data-cleaning/cache/year_to_load.txt"

# Read the year from the file
with open(year_file_path, "r") as file:
    year_to_load = file.read().strip()  # .strip() removes any surrounding whitespace or newlines

# Create a suffix based on the `to_sample` flag and sample_size_divisor
if to_sample:
    suffix = f"_sampled_{sample_size_divisor}_"
else:
    suffix = "_full_"

Parallelization: True 



In [9]:
import pandas as pd
import os
import numpy as np
from grouper import seeker
from multiprocessing import Pool, cpu_count
import traceback
import swifter
import traceback
import sys
import io
import pyarrow
import gc

In [10]:
# # Construct the file path for the Feather file
# feather_file_path = f"~/drg-pipeline/data-cleaning/data/checkpoints/checkpoint_7_py_input/python_input_{year_to_load}{suffix}.feather"

# # Expand the `~` to the user's home directory
# feather_file_path = os.path.expanduser(feather_file_path)

# # Read the Feather file
# pandas_df = pd.read_feather(feather_file_path)

# # Print the DataFrame or process it as needed
# # print(pandas_df)
# print(pandas_df[(pandas_df['patage'] == 0) & (pandas_df['ageday'].notna())])

In [11]:
# Define the patient data as a dictionary with lists
pat = {
    'id_series': ['1'],
    'patage': [0],
    'patsex': ['F'],
    'date_adm': ['2018-08-01 14:46:00'],
    'date_dis': ['2018-08-04 07:00:00'],
    'pdx': ['J189'],
    'sdx1': [None],
    'sdx2': [None],
    'sdx3': [None],
    'sdx4': [None],
    'sdx5': [None],
    'sdx6': [None],
    'sdx7': [None],
    'sdx8': [None],
    'sdx9': [None],
    'sdx10': [None],
    'sdx11': [None],
    'sdx12': [None],
    'proc1': [None],
    'proc2': [None],
    'proc3': [None],
    'proc4': [None],
    'proc5': [None],
    'proc6': [None],
    'proc7': [None],
    'proc8': [None],
    'proc9': [None],
    'proc10': [None],
    'proc11': [None],
    'proc12': [None],
    'proc13': [None],
    'proc14': [None],
    'proc15': [None],
    'proc16': [None],
    'proc17': [None],
    'proc18': [None],
    'proc19': [None],
    'proc20': [None],
    'discharge': [1],
    'birthweight': [2.717],
    'ageday': [1]
}
pandas_df = pd.DataFrame(pat)

In [12]:
# Convert the column types explicitly
print("Converting data types")
pandas_df['patage'] = pd.to_numeric(pandas_df['patage'], errors='coerce')
pandas_df['ageday'] = pd.to_numeric(pandas_df['ageday'], errors='coerce')
pandas_df['birthweight'] = pd.to_numeric(pandas_df['birthweight'], errors='coerce')
pandas_df['discharge'] = pandas_df['discharge'].astype('Int64')

# Convert string columns to 'string' dtype and replace NA values with None
string_columns = ['id_series', 'patsex', 'pdx', 'sdx1', 'sdx2', 'sdx3', 'sdx4', 'sdx5', 'sdx6', 'sdx7', 'sdx8', 'sdx9', 'sdx10', 'sdx11', 'sdx12',
                  'proc1', 'proc2', 'proc3', 'proc4', 'proc5', 'proc6', 'proc7', 'proc8', 'proc9', 'proc10', 'proc11', 'proc12',
                  'proc13', 'proc14', 'proc15', 'proc16', 'proc17', 'proc18', 'proc19', 'proc20', 'date_adm', 'date_dis']

print("Replacing with None")
# Replace missing values in place
pandas_df.replace([pd.NA, np.nan, '<NA>', 'None', 'NA', -2147483648], None, inplace=True)

print("Converting to string")
# Convert columns to string dtype after replacing the values
pandas_df[string_columns] = pandas_df[string_columns].astype('string')

print("Replacing -2147483648 with None")
# Replace -2147483648 with None again (in case it was missed)
pandas_df.replace(-2147483648, None, inplace=True)

# print("Filter discharge")
# # Filter rows where 'discharge' is not in [1, 2, 3, 4, 9]
# not_in_list_values = pandas_df.loc[~pandas_df['discharge'].isin([1, 2, 3, 4, 9]), 'discharge']

# # Get unique values and their counts
# unique_not_in_list_values = not_in_list_values.value_counts()

# # Print the unique values and their counts
# print(unique_not_in_list_values)

print("Generating info()")
pandas_df.info()
print(pandas_df)

Converting data types
Replacing with None
Converting to string
Replacing -2147483648 with None
Generating info()
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1 entries, 0 to 0
Data columns (total 41 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   id_series    1 non-null      string 
 1   patage       1 non-null      int64  
 2   patsex       1 non-null      string 
 3   date_adm     1 non-null      string 
 4   date_dis     1 non-null      string 
 5   pdx          1 non-null      string 
 6   sdx1         0 non-null      string 
 7   sdx2         0 non-null      string 
 8   sdx3         0 non-null      string 
 9   sdx4         0 non-null      string 
 10  sdx5         0 non-null      string 
 11  sdx6         0 non-null      string 
 12  sdx7         0 non-null      string 
 13  sdx8         0 non-null      string 
 14  sdx9         0 non-null      string 
 15  sdx10        0 non-null      string 
 16  sdx11        0 non-null  

In [13]:
print(pandas_df[(pandas_df['patage'] == 0) & (pandas_df['ageday'].notna())])

  id_series  patage patsex             date_adm             date_dis   pdx  \
0         1       0      F  2018-08-01 14:46:00  2018-08-04 07:00:00  J189   

   sdx1  sdx2  sdx3  sdx4  ... proc14 proc15 proc16 proc17 proc18 proc19  \
0  <NA>  <NA>  <NA>  <NA>  ...   <NA>   <NA>   <NA>   <NA>   <NA>   <NA>   

  proc20 discharge birthweight ageday  
0   <NA>         1       2.717      1  

[1 rows x 41 columns]


In [14]:
# # Sample 100,000 rows from the DataFrame
# full_df = pandas_df
# pandas_df = pandas_df.sample(n=100000, random_state=42)

In [15]:
# # SINGLE THREADED-VERSION
# print("Initializing Libraries")
# # Initialize the necessary libraries
# libs = seeker.Libraries()

# # Define a function to instantiate a Patient object for each row
# def process_patient(row, libs):
#     try:
#         # Convert the row to a dictionary and create a Patient object
#         patient = seeker.Patient(row.to_dict(), libs)
        
#         # Extract relevant attributes from the Patient object
#         result = {
#             'mdc': patient.mdc,
#             'pdc': patient.pdc,
#             'pccl': patient.pccl,
#             'drg': patient.drg,
#             'error_code': patient.error_code,
#             'warning_code': patient.warning_code
#         }
        
#         return pd.Series(result)
    
#     except Exception as e:
#         # Log the error and row information for debugging
#         print(f'''Error processing patient with id_series {row['id_series']}: {e}''')
        
#         # Optionally, you can log more information such as row content or traceback
#         traceback.print_exc()  # Print the full stack trace for more details
        
#         # Return None or default values for the error case
#         return pd.Series({
#             'mdc': None,
#             'pdc': None,
#             'pccl': None,
#             'drg': None,
#             'error_code': None,
#             'warning_code': None
#         })

# print("swifter.apply process_patient")
# # Apply the Patient class directly to each row using swifter
# pandas_df[['mdc', 'pdc', 'pccl', 'drg', 'error_code', 'warning_code']] = pandas_df.apply(
#     lambda row: process_patient(row, libs),
#     axis=1
# )
# print("Renaming columns")
# # Store the result in output to be retrieved by R
# output = pandas_df.rename(columns={'drg': 'py_drg'})
# print("Reordering columns")
# # Define the desired column order
# desired_columns = [
#     'id_series', 'mdc', 'pdc', 
#     'pccl', 'py_drg', 'error_code', 
#     'warning_code'
# ]
# print("Subsetting columns")
# # Reorder the DataFrame and drop any columns not in the desired list
# output = output[desired_columns]
# print(output)

In [16]:
# Define a function to instantiate a Patient object for each row
# def process_patient(row, libs):
#     try:
#         # Convert the row to a dictionary and create a Patient object
#         patient = seeker.Patient(row.to_dict(), libs)

#         # Extract relevant attributes from the Patient object
#         result = {
#             'mdc': patient.mdc,
#             'pdc': patient.pdc,
#             'pccl': patient.pccl,
#             'drg': patient.drg,
#             'error_code': patient.error_code,
#             'warning_code': patient.warning_code
#         }

#         return result

#     except Exception as e:
#         # Log the error and row information for debugging
#         print(f"Error processing patient with id_series {row['id_series']}: {e}")
#         traceback.print_exc()

#         # Return default values for error cases
#         return {
#             'mdc': None,
#             'pdc': None,
#             'pccl': None,
#             'drg': None,
#             'error_code': None,
#             'warning_code': None
#         }

# # Initialize the necessary libraries
# print("Initializing Libraries")
# libs = seeker.Libraries()

# # Split DataFrame into chunks for multiprocessing
# num_cores = cpu_count()  # Automatically detect the number of CPU cores
# chunks = np.array_split(pandas_df, num_cores)  # Split the DataFrame into chunks

# print(f"Processing using {num_cores} cores")

# # Use multiprocessing to process each chunk in parallel
# with Pool(num_cores) as pool:
#     results = pool.starmap(process_chunk, [(chunk, libs) for chunk in chunks])

# # Combine the results back into a single DataFrame
# processed_df = pd.concat(results)

# # Add the processed columns to the original DataFrame
# pandas_df[['mdc', 'pdc', 'pccl', 'drg', 'error_code', 'warning_code']] = processed_df

# # Rename and reorder columns
# pandas_df.rename(columns={'drg': 'py_drg'}, inplace=True)
# desired_columns = [
#     'id_series', 'mdc', 'pdc',
#     'pccl', 'py_drg', 'error_code',
#     'warning_code'
# ]
# # Drop columns not in the desired list
# columns_to_drop = [col for col in pandas_df.columns if col not in desired_columns]
# pandas_df.drop(columns=columns_to_drop, inplace=True)
# Initialize the necessary libraries

In [17]:
print(pandas_df)

  id_series  patage patsex             date_adm             date_dis   pdx  \
0         1       0      F  2018-08-01 14:46:00  2018-08-04 07:00:00  J189   

   sdx1  sdx2  sdx3  sdx4  ... proc14 proc15 proc16 proc17 proc18 proc19  \
0  <NA>  <NA>  <NA>  <NA>  ...   <NA>   <NA>   <NA>   <NA>   <NA>   <NA>   

  proc20 discharge birthweight ageday  
0   <NA>         1       2.717      1  

[1 rows x 41 columns]


In [ ]:
# Define a function to instantiate a Patient object for each row
def process_patient(row, libs):
    try:
        # Convert the row to a dictionary and create a Patient object
        patient = seeker.Patient(row.to_dict(), libs)

        # Extract relevant attributes from the Patient object
        result = {
            # 'mdc': patient.mdc,
            'pdc': patient.pdc,
            'pccl': patient.pccl,
            'drg': patient.drg,
            'error_code': patient.error_code,
            'warning_code': patient.warning_code
        }

        return result

    except Exception as e:
        # Log the error and row information for debugging
        print(f"Error processing patient with id_series {row.get('id_series', 'Unknown')}: {e}")
        traceback.print_exc()

        # Return default values for error cases
        return {
            # 'mdc': None,
            'pdc': None,
            'pccl': None,
            'drg': None,
            'error_code': None,
            'warning_code': None
        }

# Function to process a chunk of the DataFrame
def process_chunk(chunk, libs):
    return chunk.apply(lambda row: pd.Series(process_patient(row, libs)), axis=1)

# Initialize the necessary libraries
print("Initializing Libraries")
libs = seeker.Libraries()

# Split DataFrame into chunks for multiprocessing
num_cores = cpu_count()  # Automatically detect the number of CPU cores
chunks = np.array_split(pandas_df, num_cores)  # Split the DataFrame into chunks

print(f"Processing using {num_cores} cores")

# Use multiprocessing to process each chunk in parallel
with Pool(num_cores) as pool:
    results = pool.starmap(process_chunk, [(chunk, libs) for chunk in chunks])

# Combine the results back into a single DataFrame
processed_df = pd.concat(results, ignore_index=True)

# Free memory for intermediate results
del results
gc.collect()

# Add the processed columns to the original DataFrame
pandas_df[['pdc', 'pccl', 'drg', 'error_code', 'warning_code']] = processed_df

# Free up memory for the processed DataFrame
del processed_df
gc.collect()

# Rename and reorder columns
pandas_df.rename(columns={'drg': 'py_drg'}, inplace=True)

desired_columns = [
    'id_series', 'pdc',
    'pccl', 'py_drg', 'error_code',
    'warning_code'
]
# Keep only the desired columns
pandas_df = pandas_df[desired_columns]

# Final output
print("Processing complete. Output DataFrame:")
print(pandas_df.head())

Initializing Libraries
Processing using 8 cores


Error processing patient with id_series nan: 'losd'


Traceback (most recent call last):
  File "/tmp/ipykernel_194907/1843500642.py", line 5, in process_patient
    patient = seeker.Patient(row.to_dict(), libs)
              ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/resurreccion_cmc_gmail_com/drg-pipeline/data-cleaning/grouper/seeker.py", line 46, in __init__
    setattr(self, field, patient[field])
                         ~~~~~~~^^^^^^^
KeyError: 'losd'


Error processing patient with id_series nan: 'losd'


Traceback (most recent call last):
  File "/tmp/ipykernel_194907/1843500642.py", line 5, in process_patient
    patient = seeker.Patient(row.to_dict(), libs)
              ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/resurreccion_cmc_gmail_com/drg-pipeline/data-cleaning/grouper/seeker.py", line 46, in __init__
    setattr(self, field, patient[field])
                         ~~~~~~~^^^^^^^
KeyError: 'losd'


...

Error processing patient with id_series nan: 'losd'


Traceback (most recent call last):
  File "/tmp/ipykernel_194907/1843500642.py", line 5, in process_patient
    patient = seeker.Patient(row.to_dict(), libs)
              ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/resurreccion_cmc_gmail_com/drg-pipeline/data-cleaning/grouper/seeker.py", line 46, in __init__
    setattr(self, field, patient[field])
                         ~~~~~~~^^^^^^^
KeyError: 'losd'


Error processing patient with id_series nan: 'losd'


Traceback (most recent call last):
  File "/tmp/ipykernel_194907/1843500642.py", line 5, in process_patient
    patient = seeker.Patient(row.to_dict(), libs)
              ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/resurreccion_cmc_gmail_com/drg-pipeline/data-cleaning/grouper/seeker.py", line 46, in __init__
    setattr(self, field, patient[field])
                         ~~~~~~~^^^^^^^
KeyError: 'losd'


ValueError: Columns must be same length as key

In [ ]:
print(pandas_df)

In [ ]:
# Construct the file path
file_path = f"/home/resurreccion_cmc_gmail_com/drg-pipeline/data-cleaning/data/checkpoints/checkpoint_8_py_output/python_output_{year_to_load}{suffix}.feather"
# Save the DataFrame as a Feather file
pandas_df.to_feather(file_path)